In [20]:
import math
import random
import matplotlib.pyplot as plt

首先导入必备的算法库

In [21]:
def euclidean_distance(city1,city2):
    return math.hypot(city1[0]-city2[0],city1[1]-city2[1])

工具函数：（1）计算两城市指尖的欧氏距离。

In [22]:
def compute_total_distance(tour,dist_matrix):
    total=0.0
    n=len(tour)
    for i in range(n):
        total+=dist_matrix[tour[i]][tour[(i+1)%n]]
    return total

工具函数：（2）计算一条路径（哈密顿回路）的总长度。

In [23]:
def construct_tour(pheromone,eta,alpha,beta):
    n=len(pheromone)
    start=random.randint(0,n-1)
    tour=[start]
    unvisited=[i for i in range(n) if i!=start]
    current=start
    while unvisited:
        weights=[]
        for j in unvisited:
            weight=(pheromone[current][j]**alpha)*(eta[current][j]**beta)
            weights.append(weight)
        total_weight=sum(weights)
        if total_weight==0:
            probs=[1.0/len(unvisited)]*len(unvisited)
        else:
            probs=[w/total_weight for w in weights]
        next_city=random.choices(unvisited,weights=probs,k=1)[0]
        tour.append(next_city)
        unvisited.remove(next_city)
        current=next_city
    return tour

蚁群算法工具：（1）一只蚂蚁构造一条完整路径，使用信息素、启发信息、alpha、beta通过轮盘赌选择下一个城市。
param pheromone：信息素矩阵（n*n）
param eta：启发信息矩阵（1\distance）
param alpha：信息素权重
param beta：启发信息权重
return：路径列表

In [24]:
def ant_colony_optimization(dist_matrix,num_ants=20,alpha=1.0,beta=2.0,rho=0.5,Q=100,iterations=200):
    n=len(dist_matrix)
    pheromone=[[0.1 for _ in range(n)]for _ in range(n)]
    eta=[[0.0 for _ in range(n)]for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i!=j:
                eta[i][j]=1.0/dist_matrix[i][j]
            else:
                eta[i][j]=0.0
    best_tour=None
    best_length=float('inf')
    length_history=[]
    for iteration in range(iterations):
        all_tours=[]
        all_lengths=[]
        for ant in range(num_ants):
            tour=construct_tour(pheromone,eta,alpha,beta)
            length=compute_total_distance(tour,dist_matrix)
            all_tours.append(tour)
            all_lengths.append(length)
            if length<best_length:
                best_length=length
                best_tour=tour[:]
        for i in range(n):
            for j in range(n):
                pheromone[i][j]*=(1-rho)
        for ant_idx in range(num_ants):
            tour=all_tours[ant_idx]
            length=all_lengths[ant_idx]
            delta=Q/length
            for k in range(len(tour)-1):
                u,v=tour[k],tour[k+1]
                pheromone[u][v]+=delta
                pheromone[v][u]+=delta
            u,v=tour[-1],tour[0]
            pheromone[u][v]+=delta
            pheromone[v][u]+=delta
        length_history.append(best_length)
        if (iteration+1)%50==0:
            print(f"迭代次数：{iteration+1} 当前最优：{best_length:.4f}")
    return best_tour,best_length,length_history

In [25]:
def main():
    num_cities=20
    random.seed(100)
    cities=[(random.randint(0,1000),random.randint(0,1000)) for i in range(num_cities)]
    dist_matrix=[[0.0]*num_cities for _ in range(num_cities)]
    for i in range(num_cities):
        for j in range(num_cities):
            dist_matrix[i][j]=euclidean_distance(cities[i],cities[j])
    num_ants=20
    alpha=1.0
    beta=2.0
    rho=0.5
    Q=100
    iterations=200
    best_tour,best_length,history=ant_colony_optimization(dist_matrix,num_ants=num_ants,alpha=alpha,
                                                          beta=beta,rho=rho,Q=Q,iterations=iterations)
    print(f"城市数目：{num_cities}")
    print(f"最优路径：{best_tour}")
    print(f"最优距离：{best_length:.4f}")
if __name__=="__main__":
    main()

迭代次数：50 当前最优：3769.5467
迭代次数：100 当前最优：3769.5467
迭代次数：150 当前最优：3769.5467
迭代次数：200 当前最优：3769.5467
城市数目：20
最优路径：[15, 2, 6, 16, 12, 18, 11, 4, 3, 5, 19, 1, 14, 8, 10, 0, 17, 13, 9, 7]
最优距离：3769.5467
